In [1]:
import time
import ft4222
from ft4222.GPIO import Dir, Port


In [5]:

# open device with default description 'FT4222 D'
gpio = ft4222.openByDescription('FT4222 D')


# use GPIO2 as gpio (not suspend out)
gpio.setSuspendOut(False)
# use GPIO3 as gpio (not wakeup)
gpio.setWakeUpInterrupt(False)

# init GPIO2 as output
gpio.gpio_Init(gpio2 = Dir.OUTPUT, gpio3 = Dir.OUTPUT)
gpio.gpio_Write(Port.P2, True)

FT2XXDeviceError: DEVICE_NOT_OPENED

In [ ]:

for i in range(0,5):
    gpio.gpio_Write(Port.P3, True)
    time.sleep(0.5)
    # generate a square wave signal with GPIO2
    gpio.gpio_Write(Port.P3, False)
    time.sleep(0.5)

In [ ]:
gpio.close()

In [2]:
lista = ft4222.createDeviceInfoList()
print(lista)


4


In [3]:
try:
    info = ft4222.getDeviceInfoDetail(0)
    print(info)
except ft4222.FT2XXDeviceError as e:
    print('Unable to identify FT4222 device.')

{'index': 0, 'flags': 1, 'type': 3, 'id': 0, 'location': 0, 'serial': b'', 'description': b'', 'handle': 0}


In [ ]:
devaA = ft4222.openB

AttributeError: module 'ft4222' has no attribute 'openByIndex'

In [ ]:
from ft4222.SPI import Cpha, Cpol
from ft4222.SPIMaster import Mode, Clock, SlaveSelect
from ft4222.GPIO import Port, Dir
from time import sleep

# open 'device' with default description 'FT4222 A'
devA = ft4222.openByDescription('FT4222 A')

# init spi master
devA.spiMaster_Init(mode=Mode.SINGLE,
                    clock=Clock.DIV_32,  # 1.875 MHz
                    cpol= Cpol.IDLE_LOW,
                    cpha=Cpha.CLK_LEADING,
                    ssoMap=SlaveSelect.SS1)

# set port0 1 (-> note this is *not* the spi chip select, the chip select (SS0) is generated by the spi core)
#devB.gpio_Write(Port.P0, 1)
std_diagnostic = int('0000000000000001',2)
std_diag = b'\x01'

In [ ]:
devA.close()
#dev_b = ft4222.openByDescription('FT4222 B')
#dev_b.close()

In [ ]:


devA.spiMaster_SingleRead()

In [ ]:
std_diagnostic

In [ ]:
a = 0b1 << 0
b = 0b1 << 1
c = 0b11 << 0
d = 1

print(d.to_bytes(2, 'big'))
print(a | b)
print(c & a)

In [3]:
import ft4222

# Count devices
num_devs = ft4222.createDeviceInfoList()
print(f"Devices found: {num_devs}")

for i in range(num_devs):
    try:
        # Attempt to open the device to "wake" the handle
        dev = ft4222.openByLocation(i)
        detail = ft4222.getDeviceInfoDetail(i)
        print(f"Index {i} Details: {detail}")
        dev.close()
    except Exception as e:
        print(f"Could not open index {i}: {e}")

Devices found: 4
Could not open index 0: DEVICE_NOT_FOUND
Could not open index 1: DEVICE_NOT_FOUND
Could not open index 2: DEVICE_NOT_FOUND
Could not open index 3: DEVICE_NOT_FOUND


In [1]:
import ft4222

# Open the device
dev = ft4222.openByDescription("FT4222 A")

# Check the version and current mode
version = dev.getVersion()
print(f"Chip Version: {version}")

# Try to get the clock - if this works, the vendor commands ARE supported
# If this fails with VENDER_CMD_NOT_SUPPORTED, the chip is in a locked/wrong state
try:
    clk = dev.getClock()
    print(f"System Clock: {clk} MHz")
except Exception as e:
    print(f"Failed to get clock: {e}")

AttributeError: 'ft4222.ft4222.FT4222' object has no attribute 'getVersion'

In [3]:
options = dir(dev)
print(options[25:])

['__sizeof__', '__str__', '__subclasshook__', 'chipReset', 'chipRevision', 'chipVersion', 'close', 'cyclePort', 'getChipMode', 'getClock', 'getLatencyTimer', 'getMaxTransferSize', 'gpio_GetTriggerStatus', 'gpio_Init', 'gpio_Read', 'gpio_ReadTriggerQueue', 'gpio_SetInputTrigger', 'gpio_Wait', 'gpio_Write', 'i2cMaster_GetStatus', 'i2cMaster_Init', 'i2cMaster_Read', 'i2cMaster_ReadEx', 'i2cMaster_Reset', 'i2cMaster_Write', 'i2cMaster_WriteEx', 'i2cSlave_GetAddress', 'i2cSlave_GetRxStatus', 'i2cSlave_Init', 'i2cSlave_Read', 'i2cSlave_Reset', 'i2cSlave_SetAddress', 'i2cSlave_SetClockStretch', 'i2cSlave_SetRespWord', 'i2cSlave_Write', 'libVersion', 'purge', 'resetDevice', 'resetPort', 'setClock', 'setLatencyTimer', 'setSuspendOut', 'setTimeouts', 'setWakeUpInterrupt', 'spiMaster_EndTransaction', 'spiMaster_Init', 'spiMaster_MultiReadWrite', 'spiMaster_SetLines', 'spiMaster_SingleRead', 'spiMaster_SingleReadWrite', 'spiMaster_SingleWrite', 'spiSlave_GetRxStatus', 'spiSlave_Init', 'spiSlave_In

In [7]:
dev.libVersion

17041152

In [ ]:
import ft4222
import ft4222.SPIMaster as SPIM
from ft4222.SPI import Cpha, Cpol
try:
    # 1. Open all three handles
    dev0 = ft4222.openByDescription(b'FT4222 A') # Controls SS0
    dev0.resetDevice()
    version = dev0.chipVersion
    print(f"Chip Version: {version}")

    # Check if the device is actually 'functional' before Init
    print(f"Device Type: {dev0.getChipMode()}")
    #    dev1 = ft4222.openByDescription(b'FT4222 B') # Controls SS1
 #   dev2 = ft4222.openByDescription(b'FT4222 C') # Controls SS2

    # 2. Initialize the MASTER ENGINE (Only needs to be done once)
    # We do this on dev0, but it configures the shared SPI clock/data lines
    dev0.spiMaster_Init(
        SPIM.Mode.SINGLE, 
        SPIM.Clock.DIV_8, 
        Cpol.IDLE_LOW, 
        Cpha.CLK_LEADING, 
        SPIM.SlaveSelect.SS0
    )

    # 3. Use the specific handle to talk to the specific slave
    # To talk to Slave 0 (Pin SS0):
    data_from_0 = dev0.spiMaster_SingleReadWrite(bytes([0x9F, 0x00]), True)

    # To talk to Slave 1 (Pin SS1):
  #  data_from_1 = dev1.spiMaster_SingleReadWrite(bytes([0x9F, 0x00]), True)

    # To talk to Slave 2 (Pin SS2):
   # data_from_2 = dev2.spiMaster_SingleReadWrite(bytes([0x9F, 0x00]), True)
finally:
    dev0.close()
    #dev1.close()
    #dev2.close()

Chip Version: 1109525504
Device Type: 1


FT4222DeviceError: OTHER_ERROR

: 

In [7]:
dev0.close()
dev1.close()
dev2.close()

FT4222DeviceError: INVALID_HANDLE

In [1]:
from plfluidics.drivers.ft4222_hub import FT4222Hub
ft_hub = None
ft_hub = FT4222Hub()

In [2]:
ft_hub.detectDevices()
print(ft_hub.num_devices)
ft_hub.test()

4
test


In [11]:
spiA = ft_hub.initSPIDevice('FT4222 A')
spiB = ft_hub.initSPIDevice('FT4222 B')
spiC = ft_hub.initSPIDevice('FT4222 C')

In [15]:
gpio = ft_hub.initGPIODevice('FT4222 D', outputs=[2,3])

FT4222DeviceError: OTHER_ERROR

In [23]:
#gpio.write(3, True)


In [24]:
ft_hub.subunits

{'FT4222 D': <plfluidics.drivers.ft4222_hub.FT4222GPIODevice at 0x153f3e5bd90>}

In [4]:
spiA = ft_hub.initSPIDevice('FT4222 A')
spiB = ft_hub.initSPIDevice('FT4222 B')
spiC = ft_hub.initSPIDevice('FT4222 C')

FT4222DeviceError: IO_ERROR

In [26]:
#spiA.read()

In [13]:
from plfluidics.drivers.drv81008 import DRV81008_FT4222

In [14]:
drvA = DRV81008_FT4222(spiA)
drvB = DRV81008_FT4222(spiB)
drvC = DRV81008_FT4222(spiC)

In [29]:
print(drvA._temp_out)
print(drvA._temp_in)

0
0


In [15]:
ft_hub.subunits

{'FT4222 A': <plfluidics.drivers.ft4222_hub.FT4222SPIDevice_Single at 0x1e5ea8cf890>,
 'FT4222 B': <plfluidics.drivers.ft4222_hub.FT4222SPIDevice_Single at 0x1e5ea8cf9d0>,
 'FT4222 C': <plfluidics.drivers.ft4222_hub.FT4222SPIDevice_Single at 0x1e5ea2cf820>}

In [16]:
drvA.cmdReadStdDiag()


In [17]:
print(drvA._temp_out)
print(drvA._temp_in)

32513
7168


In [20]:
drvA.cmdWriteAddr(drvA.addr_en,2)

In [ ]:
drvA.cmdReadStdDiag()


In [ ]:
drvA.readRegisters()

In [1]:
#from plfluidics.hardware.valve_controller import ValveControllerPLRD1
from plfluidics.hardware.valve import ValvePLRD1
from plfluidics.drivers.ft4222_hub import FT4222Hub
from plfluidics.drivers.drv81008 import DRV81008_FT4222
ft_hub = None
ft_hub = FT4222Hub()
ft_hub.detectDevices()
print(ft_hub.num_devices)
ft_hub.test()

drvA = DRV81008_FT4222(ft_hub.initSPIDevice('FT4222 A'))

4
test


In [5]:
valve = ValvePLRD1(USB_device=drvA, address = 4)

In [7]:
valve._writeState(True)